# Mascarade Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Mascarade models on Google Colab with free GPU/TPU resources.

## Setup

### Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q huggingface-hub


### Authenticate with Hugging Face

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


### Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Data Preparation

### Download Dataset

In [ ]:
!git clone https://github.com/yourusername/mascarade.git
%cd mascarade

# Or upload your custom dataset
domain = "stm32"  # @param ["stm32", "kicad", "spice", "freecad", "dsp", "platformio", "power", "emc", "embedded", "components", "generic"]
!python download_datasets.py --domain {domain}


### Convert to Training Format

In [ ]:
import json

SEPARATOR = "=" * 80
SYSTEM_INSTRUCTIONS = {
    "kicad": "You are an expert KiCad and PCB design assistant.",
    "stm32": "You are an expert STM32 embedded systems programmer.",
    "spice": "You are an expert circuit simulation engineer.",
    "freecad": "You are an expert FreeCAD and mechanical design engineer.",
    "dsp": "You are an expert digital signal processing engineer.",
    "platformio": "You are an expert PlatformIO and embedded development specialist.",
    "power": "You are an expert power electronics engineer.",
    "emc": "You are an expert EMC engineer.",
    "embedded": "You are an expert embedded systems engineer.",
    "components": "You are an expert electronic components specialist.",
    "generic": "You are an expert programmer.",
}

def convert_to_training_format(dataset_path, output_path, domain):
    with open(dataset_path, "r") as f:
        content = f.read()

    samples = [s.strip() for s in content.split(SEPARATOR) if s.strip()]
    system_instruction = SYSTEM_INSTRUCTIONS[domain]

    training_data = []
    for sample in samples:
        if "### Question:" in sample and "### Answer:" in sample:
            parts = sample.split("### Answer:")
            question = parts[0].replace("### Question:", "").strip()
            answer = parts[1].strip()
        elif "### Instruction:" in sample and "### Code:" in sample:
            parts = sample.split("### Code:")
            question = parts[0].replace("### Instruction:", "").strip()
            answer = parts[1].strip()
        else:
            continue

        training_data.append({
            "messages": [
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer}
            ]
        })

    with open(output_path, "w") as f:
        json.dump(training_data, f, indent=2)

    return training_data

dataset_path = f"./datasets/{domain}_dataset.txt"
output_path = f"./datasets/{domain}_training.json"
training_data = convert_to_training_format(dataset_path, output_path, domain)
print(f"Converted {len(training_data)} examples")


## Model Loading

### Load Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-v0.1"  # @param ["mistralai/Mistral-7B-v0.1", "mistralai/Mistral-7B-Instruct-v0.1", "google/gemma-7b", "google/gemma-7b-it"]

# Quantization config for 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


## Fine-Tuning

### Configure Training

In [ ]:
from trl import SFTTrainer
from peft import LoraConfig
from datasets import Dataset

# Load dataset
dataset = Dataset.from_json(output_path)

# LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# Training arguments
training_args = {
    "output_dir": f"./results/{domain}_finetune",
    "num_train_epochs": 3,  # @param {type:"slider", min:1, max:10, step:1}
    "per_device_train_batch_size": 4,  # @param {type:"slider", min:1, max:8, step:1}
    "gradient_accumulation_steps": 4,
    "optim": "paged_adamw_32bit",
    "save_steps": 100,
    "logging_steps": 10,
    "learning_rate": 2e-4,
    "weight_decay": 0.001,
    "fp16": True,
    "bf16": False,
    "max_grad_norm": 0.3,
    "max_steps": -1,
    "warmup_ratio": 0.03,
    "group_by_length": True,
    "lr_scheduler_type": "cosine",
    "report_to": "none"
}


### Start Training

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

# Start training
print("Starting training...")
trainer.train()

# Save model
output_dir = f"./fine_tuned_models/{domain}_mistral_colab"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")


## Evaluation

### Test the Fine-Tuned Model

In [ ]:
from transformers import pipeline

# Load fine-tuned model
pipe = pipeline(
    "text-generation",
    model=output_dir,
    tokenizer=tokenizer,
    device_map="auto"
)

# Test prompts
test_prompts = {
    "kicad": [
        "Generate a KiCad schematic for a simple LED circuit",
        "Write Python code to create a custom footprint in KiCad"
    ],
    "stm32": [
        "Write STM32 HAL code to configure UART2",
        "Show how to read ADC values on STM32"
    ],
    "spice": [
        "Write a SPICE netlist for a common emitter amplifier",
        "Simulate an RC low-pass filter frequency response"
    ],
    "freecad": [
        "Create a FreeCAD Python script to model a gear",
        "Explain parametric modeling in FreeCAD"
    ],
    "dsp": [
        "Design a Butterworth low-pass filter at 1kHz",
        "Implement FFT in Python and plot spectrum"
    ],
    "platformio": [
        "Create PlatformIO config for ESP32 with WiFi",
        "Explain library management in PlatformIO"
    ],
    "power": [
        "Design a 12V to 5V buck converter",
        "Calculate flyback converter efficiency"
    ],
    "emc": [
        "Explain PCB layout for reducing EMI",
        "Describe CE certification testing"
    ],
    "embedded": [
        "Implement FreeRTOS task for sensor reading",
        "Explain ARM Cortex-M interrupt priority"
    ],
    "components": [
        "Recommend LM317 voltage regulator replacement",
        "Compare MOSFET vs IGBT for switching"
    ],
    "generic": [
        "Implement merge sort in Python",
        "Create a Flask API for user management"
    ]
}

print("\nTesting fine-tuned model:")
for prompt in test_prompts[domain]:
    print(f"\nPrompt: {prompt}")
    result = pipe(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    print(f"Response: {result[0]['generated_text']}")


### Save to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

repo_name = f"mascarade-{domain}-colab"  # @param {type:"string"}

# Login (if not already done)
from huggingface_hub import notebook_login
notebook_login()

# Push to Hub
api = HfApi()
api.create_repo(repo_id=repo_name, exist_ok=True)
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_name,
    repo_type="model"
)

print(f"Model uploaded to: https://huggingface.co/{repo_name}")


## Advanced Options

### Use TPU

In [ ]:
import torch_xla.core.xla_model as xm

# Initialize TPU
device = xm.xla_device()
print(f"Using device: {device}")

# Modify training args for TPU
training_args["bf16"] = True
training_args["fp16"] = False


### Resume Training

In [ ]:
# To resume from a checkpoint
from transformers import TrainingArguments

checkpoint_dir = "./results/checkpoint-100"  # @param {type:"string"}

training_args = TrainingArguments(
    output_dir=output_dir,
    **training_args,
    resume_from_checkpoint=checkpoint_dir
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()


### Merge LoRA Adapters

In [ ]:
# Merge LoRA weights with base model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, output_dir)
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(f"./merged_models/{domain}_merged")
tokenizer.save_pretrained(f"./merged_models/{domain}_merged")

print(f"Merged model saved to ./merged_models/{domain}_merged")


## Cleanup

In [ ]:
# Free memory
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()


## Notes

1. **Colab Limitations**: Free tier has ~12GB GPU memory (T4). For larger models:
   - Use gradient accumulation
   - Reduce batch size
   - Use 4-bit quantization

2. **Training Time**: Expect ~1-2 hours for 3 epochs on 7B model

3. **Cost**: Colab Pro provides better GPUs (A100) for faster training

4. **Dataset Size**: Aim for at least 100-200 high-quality examples per domain